# Data Preparation


To fully utilize the power of HftBacktest, it requires Tick-by-Tick full order book and trade feed data as input. Unfortunately, free Tick-by-Tick full order book and trade feed data for HFT is not available unlike daily bar data provided by platforms like Yahoo Finance. However, for Polymarket, we can start from prepared L2 data files directly.


## Getting started from Polymarket L2 data

Here, we read L2 data for a Polymarket market from the website.


In [1]:
import pandas as pd

slug = "btc-updown-5m-1778632800"
df = pd.read_parquet(
    f"https://s.wangshuox.com/poly_l2/{slug}.parquet",
    storage_options={"User-Agent": "Mozilla/5.0"},
)

The raw data contains book, price_change, and trade events recorded from Polymarket. First, let's take a look at the first few rows.


In [2]:
df.head()

,market_slug,condition_id,timestamp,event_type,hash,ask_prices,ask_sizes,bid_prices,bid_sizes,best_ask,best_bid,pc_price,pc_size,pc_side,new_tick_size,trade_price,trade_size,trade_side,trade_is_mirror
0,btc-updown-5m-1778632800,0x65910234de509d5e4f07b3702297ecca710f14a510c8...,2026-05-13 00:35:06.203,book,b2e24ffa0c6043d8f1534b8fcbc03b063aa69e3f,"[0.51, 0.52, 0.53, 0.54, 0.55, 0.56, 0.57, 0.5...","[233.39, 195.25, 181.32, 511.05, 262.0, 154.0,...","[0.5, 0.49, 0.48, 0.47, 0.46, 0.45, 0.44, 0.43...","[610.12, 25.0, 151.25, 193.37, 605.0, 249.0, 1...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
1,btc-updown-5m-1778632800,0x65910234de509d5e4f07b3702297ecca710f14a510c8...,2026-05-13 00:35:07.046,price_change,c56c20b96c3e2f73ed22347541968190ce662e70,None,None,None,None,0.51,0.5,0.64,720.77,SELL,NaN,NaN,NaN,NaN,None
2,btc-updown-5m-1778632800,0x65910234de509d5e4f07b3702297ecca710f14a510c8...,2026-05-13 00:35:07.542,price_change,c232a4f4901c4735c0d4a4f4d28b8896ab0992fd,None,None,None,None,0.51,0.5,0.50,605.12,BUY,NaN,NaN,NaN,NaN,None
3,btc-updown-5m-1778632800,0x65910234de509d5e4f07b3702297ecca710f14a510c8...,2026-05-13 00:35:07.542,book,c232a4f4901c4735c0d4a4f4d28b8896ab0992fd,"[0.51, 0.52, 0.53, 0.54, 0.55, 0.56, 0.57, 0.5...","[233.39, 195.25, 181.32, 511.05, 262.0, 154.0,...","[0.5, 0.49, 0.48, 0.47, 0.46, 0.45, 0.44, 0.43...","[605.12, 25.0, 151.25, 193.37, 605.0, 249.0, 1...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
4,btc-updown-5m-1778632800,0x65910234de509d5e4f07b3702297ecca710f14a510c8...,2026-05-13 00:35:07.557,last_trade_price,NaN,None,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN,0.5,5.0,SELL,True


The data needs to be converted to normalized data that can be fed into HftBacktest.  
`polymarket_to_hbt` converts Polymarket L2 data into HftBacktest's data format.


In [3]:
from hftbacktest import polymarket_to_hbt

data = polymarket_to_hbt(df)

Normalized data as follows. You can find more details on [Data](https://hftbacktest.readthedocs.io/en/latest/data.html).


In [4]:
import polars as pl

pl.DataFrame(data)

ev,exch_ts,local_ts,px,qty,order_id,ival,fval
u64,i64,i64,f64,f64,u64,i64,f64
3758096387,1778632506203000000,1778632506223000000,0.01,0.0,0,0,0.0
3758096388,1778632506203000000,1778632506223000000,0.5,610.12,0,0,0.0
3758096388,1778632506203000000,1778632506223000000,0.49,25.0,0,0,0.0
3758096388,1778632506203000000,1778632506223000000,0.48,151.25,0,0,0.0
3758096388,1778632506203000000,1778632506223000000,0.47,193.37,0,0,0.0
…,…,…,…,…,…,…,…
3758096385,1778633197315000000,1778633197335000000,0.05,0.0,0,0,0.0
3758096385,1778633197315000000,1778633197335000000,0.04,0.0,0,0,0.0
3758096385,1778633197315000000,1778633197335000000,0.03,0.0,0,0,0.0


You can save the converted data as an npz file and load it directly for later backtesting.


In [5]:
import numpy as np

np.savez_compressed(f"data/{slug}.npz", data=data)